In [29]:
import numpy as np
import pandas as pd
import joblib
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, confusion_matrix
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
%run MLProject.ipynb
df = pd.read_csv('usgs_main.csv')
df1 = df.copy()
df1['time'] = pd.to_datetime(df1['time'])
df1 = df1.sort_values('time')  # KRONOLOJİK
df1 = df1.dropna(subset=['latitude','longitude','depth','mag','time'])  # EKSİK DEĞERLERİ ATIŞ
df1['lats'] = np.floor(df1['latitude']).astype(int)
df1['lons'] = np.floor(df1['longitude']).astype(int)
df1.set_index('time')

dfweek = df1.set_index('time').resample('W').apply({
    'mag':'mean',
    'latitude':'mean',
    'longitude':'mean',
    'depth':'mean',
})
dfweek = dfweek.reset_index(drop=True)
dfweek.index = dfweek.index + 1
dfweek.index.name = 'timeindex'

dfweek['futuremag'] = dfweek['mag'].shift(-1)
dfweek['futuredepth'] = dfweek['depth'].shift(-1)
dfweek['futurelat'] = dfweek['latitude'].shift(-1)
dfweek['futurelon'] = dfweek['longitude'].shift(-1)
dfweek = dfweek.dropna(subset=['futuremag','futuredepth','futurelat','futurelon'])
train = dfweek[:int((4*len(dfweek))/5)]
test = dfweek[int(4*len(dfweek)/5):]
best_model_1 = joblib.load('models/random_forest_dataset1.pkl')

In [30]:
test_predictions = best_model_1.predict(test)
test_actual = test[['futuremag', 'futuredepth', 'futurelat', 'futurelon']].dropna()
min_len = min(len(test_predictions), len(test_actual))
target_names = ['futuremag', 'futuredepth', 'futurelat', 'futurelon']
target_labels = ['Magnitude', 'Depth', 'Latitude', 'Longitude']

In [31]:
print(f"\nDATASET 1 SONUÇLARI:")
for i, (name, label) in enumerate(zip(target_names, target_labels)):
    if i < test_predictions.shape[1] and i < test_actual.shape[1]:
        target_mse = mean_squared_error(
            test_actual.iloc[:min_len, i], 
            test_predictions[:min_len, i]
        )
        target_mae = mean_absolute_error(
            test_actual.iloc[:min_len, i], 
            test_predictions[:min_len, i]
        )
        target_r2 = r2_score(
            test_actual.iloc[:min_len, i], 
            test_predictions[:min_len, i]
        )
        
        print(f"   {label:12}: MSE={target_mse:.3f}, MAE={target_mae:.3f}, R²={target_r2:.3f}")


DATASET 1 SONUÇLARI:
   Magnitude   : MSE=0.036, MAE=0.164, R²=-2.731
   Depth       : MSE=11.905, MAE=2.709, R²=-0.970
   Latitude    : MSE=1.788, MAE=1.086, R²=-0.160
   Longitude   : MSE=15.720, MAE=3.022, R²=0.144


KLASİK THRESHOLD KONTROLÜ

In [32]:
magnitude_thresholds = [2.0, 2.5, 3.0, 3.5, 4.0]

for threshold in magnitude_thresholds:
    predicted_magnitudes = test_predictions[:min_len, 0]
    actual_magnitudes = test_actual.iloc[:min_len, 0].values
    
    predicted_earthquake = (predicted_magnitudes >= threshold).astype(int)
    actual_earthquake = (actual_magnitudes >= threshold).astype(int)
    accuracy = (predicted_earthquake == actual_earthquake).mean()
    
    total_samples = len(actual_earthquake)
    actual_earthquakes = actual_earthquake.sum()
    predicted_earthquakes = predicted_earthquake.sum()
    correct_predictions = (predicted_earthquake == actual_earthquake).sum()
    
    print(f"\nBüyüklük Eşiği: {threshold}")
    print(f"Toplam örnek sayısı: {total_samples}")
    print(f"Gerçek deprem sayısı (>={threshold}): {actual_earthquakes}")
    print(f"Tahmin edilen deprem sayısı (>={threshold}): {predicted_earthquakes}")
    print(f"Doğru tahmin sayısı: {correct_predictions}")
    print(f"Doğruluk (Accuracy): {accuracy:.3f}")
    
    try:
        cm = confusion_matrix(actual_earthquake, predicted_earthquake)
        if cm.size == 4:
            tn, fp, fn, tp = cm.ravel()
            precision = tp / (tp + fp) if (tp + fp) > 0 else 0
            recall = tp / (tp + fn) if (tp + fn) > 0 else 0
            f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
            
            print(f"Kesinlik (Precision): {precision:.3f}")
            print(f"Duyarlılık (Recall): {recall:.3f}")
            print(f"F1-Score: {f1_score:.3f}")
            print(f"Confusion Matrix: TN={tn}, FP={fp}, FN={fn}, TP={tp}")
        else:
            print("Tek sınıf mevcut, detaylı metrikler hesaplanamıyor")
    except:
        print("Confusion matrix hesaplanamıyor")


Büyüklük Eşiği: 2.0
Toplam örnek sayısı: 7
Gerçek deprem sayısı (>=2.0): 0
Tahmin edilen deprem sayısı (>=2.0): 0
Doğru tahmin sayısı: 7
Doğruluk (Accuracy): 1.000
Tek sınıf mevcut, detaylı metrikler hesaplanamıyor

Büyüklük Eşiği: 2.5
Toplam örnek sayısı: 7
Gerçek deprem sayısı (>=2.5): 0
Tahmin edilen deprem sayısı (>=2.5): 0
Doğru tahmin sayısı: 7
Doğruluk (Accuracy): 1.000
Tek sınıf mevcut, detaylı metrikler hesaplanamıyor

Büyüklük Eşiği: 3.0
Toplam örnek sayısı: 7
Gerçek deprem sayısı (>=3.0): 0
Tahmin edilen deprem sayısı (>=3.0): 0
Doğru tahmin sayısı: 7
Doğruluk (Accuracy): 1.000
Tek sınıf mevcut, detaylı metrikler hesaplanamıyor

Büyüklük Eşiği: 3.5
Toplam örnek sayısı: 7
Gerçek deprem sayısı (>=3.5): 0
Tahmin edilen deprem sayısı (>=3.5): 0
Doğru tahmin sayısı: 7
Doğruluk (Accuracy): 1.000
Tek sınıf mevcut, detaylı metrikler hesaplanamıyor

Büyüklük Eşiği: 4.0
Toplam örnek sayısı: 7
Gerçek deprem sayısı (>=4.0): 0
Tahmin edilen deprem sayısı (>=4.0): 0
Doğru tahmin sayısı: 

c:\Users\eren1\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:407: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
c:\Users\eren1\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:407: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
c:\Users\eren1\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:407: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
c:\Users\eren1\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:407: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct s

KONUM KONTROLÜ

In [33]:
test_with_predictions = test.iloc[:min_len].copy()
test_with_predictions['predicted_mag'] = test_predictions[:min_len, 0]
test_with_predictions['predicted_lat'] = test_predictions[:min_len, 2]
test_with_predictions['predicted_lon'] = test_predictions[:min_len, 3]

# Daha büyük bölge grupları oluştur (1.0 derece aralıklarla)
test_with_predictions['lat_group'] = np.round(test_with_predictions['latitude'])
test_with_predictions['lon_group'] = np.round(test_with_predictions['longitude'])
test_with_predictions['location_group'] = test_with_predictions['lat_group'].astype(str) + '_' + test_with_predictions['lon_group'].astype(str)

# Her konum grubu için değerlendirme
location_groups = test_with_predictions.groupby('location_group').size()
valid_locations = location_groups[location_groups >= 1].index  # En az 2 veri noktası

print(f"Yeterli veri olan bölge sayısı: {len(valid_locations)}")

threshold = 4.0

for location in valid_locations[:10]:
    location_data = test_with_predictions[test_with_predictions['location_group'] == location]
    lat, lon = location.split('_')
    
    # threshold'dan büyük deprem var mı bakıyoruz,sondaki analiz yapay zek yardımıyla yazıldı
    has_actual_earthquake = (location_data['futuremag'] >= threshold).any()
    has_predicted_earthquake = (location_data['predicted_mag'] >= threshold).any()
    correct_prediction = has_actual_earthquake == has_predicted_earthquake
    print(f"Bölge ({lat}°, {lon}°) - Veri sayısı: {len(location_data)}")
    print(f"  Gerçek: {'Deprem var' if has_actual_earthquake else 'Deprem yok'}")
    print(f"  Tahmin: {'Deprem var' if has_predicted_earthquake else 'Deprem yok'}")
    print(f"  Doğru tahmin: {'✓' if correct_prediction else '✗'}")
    print(f"  Max gerçek büyüklük: {location_data['futuremag'].max():.2f}")
    print(f"  Max tahmin büyüklük: {location_data['predicted_mag'].max():.2f}")
    print()

Yeterli veri olan bölge sayısı: 7
Bölge (37.0°, -108.0°) - Veri sayısı: 1
  Gerçek: Deprem yok
  Tahmin: Deprem yok
  Doğru tahmin: ✓
  Max gerçek büyüklük: 1.78
  Max tahmin büyüklük: 1.55

Bölge (37.0°, -112.0°) - Veri sayısı: 1
  Gerçek: Deprem yok
  Tahmin: Deprem yok
  Doğru tahmin: ✓
  Max gerçek büyüklük: 1.60
  Max tahmin büyüklük: 1.54

Bölge (37.0°, -114.0°) - Veri sayısı: 1
  Gerçek: Deprem yok
  Tahmin: Deprem yok
  Doğru tahmin: ✓
  Max gerçek büyüklük: 1.66
  Max tahmin büyüklük: 1.57

Bölge (37.0°, -115.0°) - Veri sayısı: 1
  Gerçek: Deprem yok
  Tahmin: Deprem yok
  Doğru tahmin: ✓
  Max gerçek büyüklük: 1.68
  Max tahmin büyüklük: 1.62

Bölge (37.0°, -117.0°) - Veri sayısı: 1
  Gerçek: Deprem yok
  Tahmin: Deprem yok
  Doğru tahmin: ✓
  Max gerçek büyüklük: 1.85
  Max tahmin büyüklük: 1.60

Bölge (38.0°, -113.0°) - Veri sayısı: 1
  Gerçek: Deprem yok
  Tahmin: Deprem yok
  Doğru tahmin: ✓
  Max gerçek büyüklük: 1.72
  Max tahmin büyüklük: 1.57

Bölge (40.0°, -118.0°) -

In [34]:
pn=pd.read_csv('Significant Earthquake Dataset 1900-2023.csv')
df2=pn
df2=df2.rename(columns={'Time':'time','Mag':'mag','Depth':'depth','Latitude':'latitude','Longitude':'longitude' })#küçük harf uyuşmazlığı olmasın diye
df2['time'] = pd.to_datetime(df2['time'])
df2 = df2.sort_values('time')
df2 = df2.dropna(subset=['latitude','longitude','depth','mag','time'])
df2['lats'] = np.floor(df2['latitude']).astype(int)
df2['lons'] = np.floor(df2['longitude']).astype(int)
dfyear = df2.set_index('time').resample('YE').apply({
    'mag':'mean',
    'latitude':'mean',
    'longitude':'mean',
    'depth':'mean',
})
dfyear = dfyear.reset_index(drop=True)
dfyear.index = dfyear.index + 1
dfyear.index.name = 'timeindex'
dfyear['futuremag']=dfyear['mag'].shift(-1)
dfyear['futuredepth']=dfyear['depth'].shift(-1)
dfyear['futurelat']=dfyear['latitude'].shift(-1)
dfyear['futurelon']=dfyear['longitude'].shift(-1)
dfyear=dfyear.dropna(subset=['futuremag','futuredepth','futurelat','futurelon'])
train_2 = dfyear[:int((4*len(dfyear))/5)]
test_2 = dfyear[int(4*len(dfyear)/5):]
best_model_2=joblib.load('models/random_forest_dataset2.pkl')


In [35]:
test_predictions_2 = best_model_2.predict(test_2)
test_actual_2 = test_2[['futuremag', 'futuredepth', 'futurelat', 'futurelon']].dropna()
min_len_2 = min(len(test_predictions_2), len(test_actual_2))

In [36]:
print(f"\nDATASET 2 SONUÇLARI:")
for i, (name, label) in enumerate(zip(target_names, target_labels)):
    if i < test_predictions_2.shape[1] and i < test_actual_2.shape[1]:
        target_mse = mean_squared_error(
            test_actual_2.iloc[:min_len_2, i], 
            test_predictions_2[:min_len_2, i]
        )
        target_mae = mean_absolute_error(
            test_actual_2.iloc[:min_len_2, i], 
            test_predictions_2[:min_len_2, i]
        )
        target_r2 = r2_score(
            test_actual_2.iloc[:min_len_2, i], 
            test_predictions_2[:min_len_2, i]
        )
        
        print(f"   {label:12}: MSE={target_mse:.3f}, MAE={target_mae:.3f}, R²={target_r2:.3f}")



DATASET 2 SONUÇLARI:
   Magnitude   : MSE=0.001, MAE=0.017, R²=-0.821
   Depth       : MSE=205.904, MAE=12.962, R²=-0.741
   Latitude    : MSE=15.185, MAE=3.078, R²=-0.271
   Longitude   : MSE=221.888, MAE=12.331, R²=-0.415


In [37]:
magnitude_thresholds_2 = [6.0, 6.5, 7.0, 7.5]

for threshold in magnitude_thresholds_2:
    predicted_magnitudes = test_predictions_2[:min_len_2, 0]
    actual_magnitudes = test_actual_2.iloc[:min_len_2, 0].values
    
    predicted_earthquake = (predicted_magnitudes >= threshold).astype(int)
    actual_earthquake = (actual_magnitudes >= threshold).astype(int)
    accuracy = (predicted_earthquake == actual_earthquake).mean()
    
    total_samples = len(actual_earthquake)
    actual_earthquakes = actual_earthquake.sum()
    predicted_earthquakes = predicted_earthquake.sum()
    correct_predictions = (predicted_earthquake == actual_earthquake).sum()
    
    print(f"\nBüyüklük Eşiği: {threshold}")
    print(f"Toplam örnek sayısı: {total_samples}")
    print(f"Gerçek deprem sayısı (>={threshold}): {actual_earthquakes}")
    print(f"Tahmin edilen deprem sayısı (>={threshold}): {predicted_earthquakes}")
    print(f"Doğru tahmin sayısı: {correct_predictions}")
    print(f"Doğruluk (Accuracy): {accuracy:.3f}")
    
    try:
        cm = confusion_matrix(actual_earthquake, predicted_earthquake)
        if cm.size == 4:
            tn, fp, fn, tp = cm.ravel()
            precision = tp / (tp + fp) if (tp + fp) > 0 else 0
            recall = tp / (tp + fn) if (tp + fn) > 0 else 0
            f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
            
            print(f"Kesinlik (Precision): {precision:.3f}")
            print(f"Duyarlılık (Recall): {recall:.3f}")
            print(f"F1-Score: {f1_score:.3f}")
            print(f"Confusion Matrix: TN={tn}, FP={fp}, FN={fn}, TP={tp}")
        else:
            print("Tek sınıf mevcut, detaylı metrikler hesaplanamıyor")
    except:
        print("Confusion matrix hesaplanamıyor")


Büyüklük Eşiği: 6.0
Toplam örnek sayısı: 20
Gerçek deprem sayısı (>=6.0): 0
Tahmin edilen deprem sayısı (>=6.0): 0
Doğru tahmin sayısı: 20
Doğruluk (Accuracy): 1.000
Tek sınıf mevcut, detaylı metrikler hesaplanamıyor

Büyüklük Eşiği: 6.5
Toplam örnek sayısı: 20
Gerçek deprem sayısı (>=6.5): 0
Tahmin edilen deprem sayısı (>=6.5): 0
Doğru tahmin sayısı: 20
Doğruluk (Accuracy): 1.000
Tek sınıf mevcut, detaylı metrikler hesaplanamıyor

Büyüklük Eşiği: 7.0
Toplam örnek sayısı: 20
Gerçek deprem sayısı (>=7.0): 0
Tahmin edilen deprem sayısı (>=7.0): 0
Doğru tahmin sayısı: 20
Doğruluk (Accuracy): 1.000
Tek sınıf mevcut, detaylı metrikler hesaplanamıyor

Büyüklük Eşiği: 7.5
Toplam örnek sayısı: 20
Gerçek deprem sayısı (>=7.5): 0
Tahmin edilen deprem sayısı (>=7.5): 0
Doğru tahmin sayısı: 20
Doğruluk (Accuracy): 1.000
Tek sınıf mevcut, detaylı metrikler hesaplanamıyor


c:\Users\eren1\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:407: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
c:\Users\eren1\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:407: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
c:\Users\eren1\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:407: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
c:\Users\eren1\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:407: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct s

In [38]:
test_with_predictions_2 = test_2.iloc[:min_len].copy()
test_with_predictions_2['predicted_mag'] = test_predictions_2[:min_len, 0]
test_with_predictions_2['predicted_lat'] = test_predictions_2[:min_len, 2]
test_with_predictions_2['predicted_lon'] = test_predictions_2[:min_len, 3]

test_with_predictions_2['lat_group'] = np.round(test_with_predictions_2['latitude'] / 5.0) * 5.0
test_with_predictions_2['lon_group'] = np.round(test_with_predictions_2['longitude'] / 5.0) * 5.0
test_with_predictions_2['location_group'] = test_with_predictions_2['lat_group'].astype(str) + '_' + test_with_predictions_2['lon_group'].astype(str)

location_groups_2 = test_with_predictions_2.groupby('location_group').size()
valid_locations_2 = location_groups_2[location_groups_2 >= 1].index  # En az 2 veri noktası

print(f"Yeterli veri olan bölge sayısı: {len(valid_locations_2)}")

threshold = 6.0  # Yıllık data için daha yüksek eşik

for location in valid_locations_2:  
    location_data = test_with_predictions_2[test_with_predictions_2['location_group'] == location]
    lat, lon = location.split('_')
    
    
    has_actual_earthquake = (location_data['futuremag'] >= threshold).any()
    has_predicted_earthquake = (location_data['predicted_mag'] >= threshold).any()
    
    # Doğru tahmin mi?
    correct_prediction = has_actual_earthquake == has_predicted_earthquake
    
    print(f"Bölge ({lat}°, {lon}°) - Veri sayısı: {len(location_data)}")
    print(f"  Gerçek: {'Deprem var' if has_actual_earthquake else 'Deprem yok'}")
    print(f"  Tahmin: {'Deprem var' if has_predicted_earthquake else 'Deprem yok'}")
    print(f"  Doğru tahmin: {'✓' if correct_prediction else '✗'}")
    print(f"  Max gerçek büyüklük: {location_data['futuremag'].max():.2f}")
    print(f"  Max tahmin büyüklük: {location_data['predicted_mag'].max():.2f}")
    print()


Yeterli veri olan bölge sayısı: 7
Bölge (-0.0°, 25.0°) - Veri sayısı: 1
  Gerçek: Deprem yok
  Tahmin: Deprem yok
  Doğru tahmin: ✓
  Max gerçek büyüklük: 5.87
  Max tahmin büyüklük: 5.86

Bölge (-0.0°, 40.0°) - Veri sayısı: 1
  Gerçek: Deprem yok
  Tahmin: Deprem yok
  Doğru tahmin: ✓
  Max gerçek büyüklük: 5.86
  Max tahmin büyüklük: 5.86

Bölge (-0.0°, 55.0°) - Veri sayısı: 1
  Gerçek: Deprem yok
  Tahmin: Deprem yok
  Doğru tahmin: ✓
  Max gerçek büyüklük: 5.88
  Max tahmin büyüklük: 5.86

Bölge (0.0°, 40.0°) - Veri sayısı: 1
  Gerçek: Deprem yok
  Tahmin: Deprem yok
  Doğru tahmin: ✓
  Max gerçek büyüklük: 5.89
  Max tahmin büyüklük: 5.86

Bölge (0.0°, 45.0°) - Veri sayısı: 1
  Gerçek: Deprem yok
  Tahmin: Deprem yok
  Doğru tahmin: ✓
  Max gerçek büyüklük: 5.86
  Max tahmin büyüklük: 5.86

Bölge (5.0°, 40.0°) - Veri sayısı: 1
  Gerçek: Deprem yok
  Tahmin: Deprem yok
  Doğru tahmin: ✓
  Max gerçek büyüklük: 5.86
  Max tahmin büyüklük: 5.86

Bölge (5.0°, 50.0°) - Veri sayısı: 1
  

TEK VERİ İÇİN TAHMİN

In [39]:
def single_prediction_rf(test_data, model, sample_index=5): 
    if sample_index >= len(test_data):
    
        return None, None
    
    # Alttaki 3 satırı AI ekledi,çünkü veri shift yapamadığı için geçmiş yoktu ve tahmin yapamıyordu.
    shiftnum = getattr(model, 'shiftnum', 3)  # Model'den shiftnum değerini al
    start_index = max(0, sample_index - shiftnum)  # En az shiftnum kadar önceki satırı da al
    end_index = sample_index + 1
    
    # Yeterli geçmiş veri var mı kontrol et
    if sample_index < shiftnum:
        return None, None
    
    # Shift için gerekli olan tüm satırları al
    sample_rows = test_data.iloc[start_index:end_index]
    target_row = test_data.iloc[sample_index:sample_index+1]  # Sadece hedef satır
    
    feature_cols = ['mag', 'depth', 'latitude', 'longitude']
    for col in feature_cols:
        print(f"  {col:12}: {target_row[col].iloc[0]:.4f}")
    
    target_cols = ['futuremag', 'futuredepth', 'futurelat', 'futurelon']
    for col in target_cols:
        print(f"  {col:12}: {target_row[col].iloc[0]:.4f}")
    
    try:
        
        
        # Tüm gerekli satırları model'e gönder (shift için)
        prediction = model.predict(sample_rows)
        
        # Model birden fazla tahmin döndürebilir, son tahmini al
        if len(prediction) > 0:
            final_prediction = prediction[-1]  # Son tahmin (bizim hedef satırımız)
            
            # Tahminleri ve hataları göster
            print(f"\n TAHMİNLER VE HATALAR:")
            target_labels = ['Future Mag', 'Future Depth', 'Future Lat', 'Future Lon']
            
            for i, (col, label) in enumerate(zip(target_cols, target_labels)):
                actual_val = target_row[col].iloc[0]
                pred_val = final_prediction[i]
                error = abs(actual_val - pred_val)
                
                print(f"  {label:12}: Tahmin={pred_val:.4f}, Gerçek={actual_val:.4f}, Hata={error:.4f}")
            
            return final_prediction.reshape(1, -1), target_row[target_cols].values
        else:
            print("Model tahmin döndürmedi!")
            return None, None
        
    except Exception as e:
        print(f"{str(e)}")#AI ın eklediği bu kod sayesinde debug edebildim.
        print(f"Debug: Model type: {type(model)}")
        return None, None


print("DATASET 1 TESTİ:")
tahmin1, gercek1 = single_prediction_rf(test, best_model_1, sample_index=5)  

print("\nDATASET 2 TESTİ:")        
tahmin2, gercek2 = single_prediction_rf(test_2, best_model_2, sample_index=5)

DATASET 1 TESTİ:
  mag         : 1.5968
  depth       : 20.3829
  latitude    : 39.6866
  longitude   : -117.9273
  futuremag   : 1.8904
  futuredepth : 23.0301
  futurelat   : 36.5783
  futurelon   : -108.0801

 TAHMİNLER VE HATALAR:
  Future Mag  : Tahmin=1.5965, Gerçek=1.8904, Hata=0.2939
  Future Depth: Tahmin=19.7140, Gerçek=23.0301, Hata=3.3161
  Future Lat  : Tahmin=37.3828, Gerçek=36.5783, Hata=0.8045
  Future Lon  : Tahmin=-112.3882, Gerçek=-108.0801, Hata=4.3081

DATASET 2 TESTİ:
  mag         : 5.8511
  depth       : 62.5755
  latitude    : -1.2222
  longitude   : 39.0798
  futuremag   : 5.8563
  futuredepth : 57.3217
  futurelat   : 1.0368
  futurelon   : 46.4125

 TAHMİNLER VE HATALAR:
  Future Mag  : Tahmin=5.8589, Gerçek=5.8563, Hata=0.0025
  Future Depth: Tahmin=75.2482, Gerçek=57.3217, Hata=17.9265
  Future Lat  : Tahmin=1.2612, Gerçek=1.0368, Hata=0.2244
  Future Lon  : Tahmin=35.0406, Gerçek=46.4125, Hata=11.3719


KLASÖR OKUMA,LSTM İLE BENZER MANTALİTE,SADECE GEREKLİ DEĞİŞİKLİKLER YAPILDI

In [40]:

import os
import glob

def create_sample_folder_rf(test_data, folder_name="demo_samples_rf", n_samples=15, shiftnum=3):
    # Klasör oluştur
    os.makedirs(folder_name, exist_ok=True)
    
    print(f"{folder_name} klasörü oluşturuluyor...")
    
    # Test verisinin uzunluğunu kontrol et
    max_start_idx = len(test_data) - shiftnum - 1
        
    # Rastgele başlangıç noktaları seçme
    np.random.seed(42)
    sample_count = min(n_samples, max_start_idx)
    start_indices = np.random.choice(max_start_idx, size=sample_count, replace=False)
    
    for i, start_idx in enumerate(start_indices):
        # shiftnum + 1 kadar ardışık satır al (shift için geçmiş + hedef)
        end_idx = start_idx + shiftnum + 1
        sample_chunk = test_data.iloc[start_idx:end_idx]
        
        filename = f"{folder_name}/sample_{i+1:02d}.csv"
        sample_chunk.to_csv(filename, index=False)
    
    print(f"{len(start_indices)} adet örnek {folder_name} klasörüne kaydedildi.")
    print(f"Her dosya {shiftnum + 1} satır içeriyor.")
    return folder_name


def predict_from_folder_rf(folder_name, model, shiftnum=3):
    #Dosya oku,tahmin yap
    csv_files = glob.glob(f"{folder_name}/*.csv")
    csv_files.sort()
    
    all_predictions = []
    all_actuals = []
    all_filenames = []
    
    print(f"\n{folder_name} klasöründeki {len(csv_files)} dosya işleniyor...")
    
    for csv_file in csv_files:
        try:
            
            sample_df = pd.read_csv(csv_file)
            # Minimum satır kontrolü
            if len(sample_df) < shiftnum + 1:
                continue
            target_cols = ['futuremag', 'futuredepth', 'futurelat', 'futurelon']
            available_targets = [col for col in target_cols if col in sample_df.columns]
            if len(available_targets) == 0:#hedef sütun yoksa(AI)
                continue
            
            # Son satırın target değerleri
            actual_values = sample_df[available_targets].iloc[-1].values
            
            # Model ile tahmin yap (tüm DataFrame'i gönder)
            try:#Yapay zeka yardımıyla yazılan bir blok
                prediction = model.predict(sample_df)
                
                if len(prediction) > 0:
                    # Son tahmini al (bizim hedef satırımız)
                    final_prediction = prediction[-1]
                    # Sonuçları kaydet
                    all_predictions.append(final_prediction)
                    all_actuals.append(actual_values)
                    all_filenames.append(os.path.basename(csv_file))
                    
                else:
                    print(f"{os.path.basename(csv_file)}: Model tahmin döndürmedi")
                    
            except Exception as pred_error:
                print(f"{os.path.basename(csv_file)}: Tahmin hatası - {str(pred_error)}")
            
        except Exception as e:
            print(f"{csv_file}: Dosya okuma hatası - {str(e)}")
    
    if len(all_predictions) > 0:
        return np.array(all_predictions), np.array(all_actuals), all_filenames
    else:
        return np.array([]), np.array([]), []


def run_demo_rf(test_data, model, model_name="RandomForest", n_samples=15):

    # Model'den shiftnum değerini aldık,bunu tek tahmin sırasında da yapmıştık
    shiftnum = getattr(model, 'shiftnum', 3)
    
    # Klasör oluştur ve örnekleri kaydet
    folder_name = f"demo_samples_rf_{model_name.lower().replace(' ', '_')}"
    create_sample_folder_rf(test_data, folder_name, n_samples, shiftnum)
    
    # Tahminleri yap
    predictions, actuals, filenames = predict_from_folder_rf(folder_name, model, shiftnum)
    
    if len(predictions) > 0:
        # Genel performans metrikleri
        target_labels = ['Magnitude', 'Depth', 'Latitude', 'Longitude']
        
        for i, label in enumerate(target_labels):
            if i < predictions.shape[1] and i < actuals.shape[1]:
                mse = mean_squared_error(actuals[:, i], predictions[:, i])
                mae = mean_absolute_error(actuals[:, i], predictions[:, i])
                r2 = r2_score(actuals[:, i], predictions[:, i])
                
                print(f"{label:<12}: MSE={mse:.3f}, MAE={mae:.3f}, R²={r2:.3f}")
    
    return predictions, actuals, filenames


def predict_single_file_rf(file_path, model):
    print("tek bir file için tahmin")
    
    try:
        # Dosyayı oku
        sample_df = pd.read_csv(file_path)
        print(f"Dosya boyutu: {len(sample_df)} satır")
        
        # Model'den shiftnum al
        shiftnum = getattr(model, 'shiftnum', 3)
        
        # Minimum satır kontrolü
        if len(sample_df) < shiftnum + 1:
            return None, None
        
        # Son satırın bilgilerini göster (tahmin hedefi)
        last_row = sample_df.iloc[-1]
        print(f"  Mevcut Büyüklük: {last_row['mag']:.3f}")
        print(f"  Mevcut Derinlik: {last_row['depth']:.3f}")
        print(f"  Mevcut Konum: ({last_row['latitude']:.3f}, {last_row['longitude']:.3f})")
        
        # Target değerleri
        target_cols = ['futuremag', 'futuredepth', 'futurelat', 'futurelon']
        available_targets = [col for col in target_cols if col in sample_df.columns]
        
        if len(available_targets) == 0:
        
            return None, None
        
        print(f"\nHEDEF DEĞERLER:")
        actual_values = []
        for col in available_targets:
            val = last_row[col]
            actual_values.append(val)
            col_name = col.replace('future', '').title()
            print(f"  Gelecek {col_name}: {val:.3f}")
        
        # Model ile tahmin yap
        prediction = model.predict(sample_df)
        
        if len(prediction) > 0:
            # Son tahmini al
            final_prediction = prediction[-1]
            
            print(f"\n MODEL TAHMİNLERİ:")
            target_labels = ['Magnitude', 'Depth', 'Latitude', 'Longitude']
            total_error = 0
            valid_predictions = 0
            
            for i, (col, label) in enumerate(zip(available_targets, target_labels)):
                if i < len(final_prediction) and i < len(actual_values):
                    pred_val = final_prediction[i]
                    actual_val = actual_values[i]
                    error = abs(actual_val - pred_val)
                    total_error += error
                    valid_predictions += 1
                    
                    print(f"  {label:<12}: Tahmin={pred_val:.3f}, Gerçek={actual_val:.3f}, Hata={error:.3f}")
            
            
            return final_prediction, np.array(actual_values)
        else:
            print("Model tahmin döndürmedi!")
            return None, None
            
    except Exception as e:
        print(f"\n Hata: {str(e)}")
        return None, None



print("\n DATASET 1 - RandomForest Model Demo")
pred_rf1, act_rf1, files_rf1 = run_demo_rf(
    test, best_model_1, "RandomForest_Dataset1", 10
)

# İlk oluşturulan klasörden bir dosya seç ve test et
sample_file = "demo_samples_rf_randomforest_dataset1/sample_01.csv"
if os.path.exists(sample_file):
    pred_single_rf, act_single_rf = predict_single_file_rf(sample_file, best_model_1)
else:
    print(f"Örnek dosya bulunamadı: {sample_file}")

# Dataset 2 için demo
print("\nDATASET 2 - RandomForest Model Demo")
pred_rf2, act_rf2, files_rf2 = run_demo_rf(
    test_2, best_model_2, "RandomForest_Dataset2", 8
)

# Dataset 2 için tek dosya testi
print("\n Dataset 2 - Klasörden tek dosya testi:")
sample_file_2 = "demo_samples_rf_randomforest_dataset2/sample_01.csv"
if os.path.exists(sample_file_2):
    pred_single_rf2, act_single_rf2 = predict_single_file_rf(sample_file_2, best_model_2)
else:
    print(f" Örnek dosya bulunamadı: {sample_file_2}")




 DATASET 1 - RandomForest Model Demo
demo_samples_rf_randomforest_dataset1 klasörü oluşturuluyor...
6 adet örnek demo_samples_rf_randomforest_dataset1 klasörüne kaydedildi.
Her dosya 3 satır içeriyor.

demo_samples_rf_randomforest_dataset1 klasöründeki 6 dosya işleniyor...
Magnitude   : MSE=0.042, MAE=0.176, R²=-3.062
Depth       : MSE=12.400, MAE=2.441, R²=-0.743
Latitude    : MSE=2.169, MAE=1.174, R²=-0.248
Longitude   : MSE=22.255, MAE=4.177, R²=-0.094
tek bir file için tahmin
Dosya boyutu: 3 satır
  Mevcut Büyüklük: 1.720
  Mevcut Derinlik: 21.887
  Mevcut Konum: (37.446, -113.507)

HEDEF DEĞERLER:
  Gelecek Mag: 1.659
  Gelecek Depth: 18.975
  Gelecek Lat: 37.308
  Gelecek Lon: -116.806

 MODEL TAHMİNLERİ:
  Magnitude   : Tahmin=1.617, Gerçek=1.659, Hata=0.042
  Depth       : Tahmin=20.034, Gerçek=18.975, Hata=1.058
  Latitude    : Tahmin=37.262, Gerçek=37.308, Hata=0.046
  Longitude   : Tahmin=-111.196, Gerçek=-116.806, Hata=5.610

DATASET 2 - RandomForest Model Demo
demo_sample